# Lab | BabyAGI with agent

**Change the planner objective below by changing the objective and the associated prompts and potential tools and agents - Wear your creativity and AI engineering hats
You can't get this wrong!**

You would need the OpenAI API KEY and the [SerpAPI KEY](https://serpapi.com/manage-api-keyhttps://serpapi.com/manage-api-key) to run this lab.


In [ ]:
#JUST TO CHECK TORCH AND CUDA INSTALLATION

import torch

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

x = torch.randn(4096, 4096, device="cuda")
y = x @ x

print("Tensor device:", y.device)


Torch version: 2.7.0.dev20250310+cu124
CUDA available: True
GPU: NVIDIA GeForce RTX 5070 Ti Laptop GPU


c:\Users\Adria\OneDrive\Desktop\Project- Baby agent\Babyagi-with-agent\gpu-env\lib\site-packages\torch\cuda\__init__.py:287: UserWarning: 
NVIDIA GeForce RTX 5070 Ti Laptop GPU with CUDA capability sm_120 is not compatible with the current PyTorch installation.
The current PyTorch install supports CUDA capabilities sm_50 sm_60 sm_61 sm_70 sm_75 sm_80 sm_86 sm_90.
If you want to use the NVIDIA GeForce RTX 5070 Ti Laptop GPU GPU with PyTorch, please check the instructions at https://pytorch.org/get-started/locally/

  warnings.warn(


RuntimeError: CUDA error: no kernel image is available for execution on the device
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [ ]:
exit()


: 

In [ ]:
exit()


## BabyAGI with Tools

This notebook builds on top of [baby agi](baby_agi.html), but shows how you can swap out the execution chain. The previous execution chain was just an LLM which made stuff up. By swapping it out with an agent that has access to tools, we can hopefully get real reliable information

## Install and Import Required Modules

In [3]:
# ============================================================
# INSTALL REQUIRED PACKAGES
# ============================================================
# We install a specific set of LangChain packages that work together.
# google-search-results is required for SerpAPI search.
!pip install -q langchain langchain-openai langchain-core langchain-community \
                  langchain-experimental langchain-classic python-dotenv \
                  faiss-cpu google-search-results


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
# PREVIOUS CODE !pip install langchain_community langchain_experimental langchain_openai

In [ ]:
# PREVIOUS CODE
#from typing import Optional
#from langchain.chains import LLMChain
#from langchain.prompts import PromptTemplate
#from langchain_experimental.autonomous_agents import BabyAGI
#from langchain_openai import OpenAI, OpenAIEmbeddings

In [9]:
# ============================================================
# IMPORTS
# ============================================================

import os
import sys
import getpass
from dotenv import load_dotenv, find_dotenv

# OpenAI + Embeddings
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

# Tools + Document schema
from langchain_core.tools import Tool
from langchain_core.documents import Document

# BabyAGI (experimental LangChain agent)
from langchain_experimental.autonomous_agents import BabyAGI

# Vector store
from langchain_community.vectorstores import FAISS

# Search tool
from langchain_community.utilities import SerpAPIWrapper

# Classic agent wrapper
from langchain_classic.agents import initialize_agent, AgentType

## Connect to the Vector Store

Depending on what vectorstore you use, this step may look different.

In [ ]:
# PREVIOUS CODE
# %pip install faiss-cpu > /dev/null
# %pip install google-search-results > /dev/null
#from langchain.docstore import InMemoryDocstore
#from langchain_community.vectorstores import FAISS

In [13]:
# ============================================================
# LOAD ENVIRONMENT VARIABLES
# ============================================================
# If .env file exists, load it. Otherwise user inputs API keys manually.

load_dotenv(find_dotenv())

# Enter API keys interactively (safer for public notebooks)
os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OPENAI_API_KEY: ")
os.environ["SERPAPI_API_KEY"] = getpass.getpass("Enter your SERPAPI_API_KEY: ")


In [15]:
# ============================================================
# INITIALIZE LLM + EMBEDDINGS
# ============================================================
# ChatOpenAI uses the OpenAI Chat Completions API.
# model="gpt-3.5-turbo" is low-cost and works well for BabyAGI examples.

llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)

# Embeddings used by FAISS for vector search
embeddings_model = OpenAIEmbeddings()

In [31]:
# # PREVIOUS CODE
#  Define your embedding model
# embeddings_model = OpenAIEmbeddings()
# Initialize the vectorstore as empty
# import faiss

# embedding_size = 1536
# index = faiss.IndexFlatL2(embedding_size)
# vectorstore = FAISS(embeddings_model.embed_query, index, InMemoryDocstore({}), {})

In [16]:
# ============================================================
# CREATE VECTORSTORE WITH EXAMPLE DOCUMENTS
# ============================================================
# BabyAGI uses a vector database to store and recall tasks.
# Here we store generic "task descriptions" as example documents.

documents = [
    Document(page_content="Create comprehensive plans", metadata={"task": "Planning"}),
    Document(page_content="Research information", metadata={"task": "Research"}),
    Document(page_content="Generate schedules", metadata={"task": "Scheduling"}),
    Document(page_content="Create checklists",  metadata={"task": "Organization"}),
]

# Build the FAISS vector index
vectorstore = FAISS.from_documents(documents, embeddings_model)


## Define the Chains

BabyAGI relies on three LLM chains:
- Task creation chain to select new tasks to add to the list
- Task prioritization chain to re-prioritize tasks
- Execution Chain to execute the tasks


NOTE: in this notebook, the Execution chain will now be an agent.

In [17]:
# ============================================================
# DEFINE TOOLS (FUNCTIONS BABYAGI CAN USE)
# ============================================================
# Each tool wraps an LLM or external call (like SerpAPI).
# Tools allow BabyAGI to perform actions.

# ---- 1. PLANNING TOOL ----
def create_plan(objective: str) -> str:
    """Generate a detailed plan for a given objective."""
    response = llm.invoke(
        f"Create a detailed step-by-step plan for: {objective}.\n"
        f"Include milestones, timeline, and required resources."
    )
    return response.content

# ---- 2. SEARCH TOOL ----
def research_topic(query: str) -> str:
    """Run a SerpAPI search or return a fallback message."""
    try:
        search = SerpAPIWrapper()
        return search.run(query)
    except:
        return f"Basic research result for: {query}"

# ---- 3. SCHEDULE TOOL ----
def generate_schedule(objective: str) -> str:
    """Generate a multi-day schedule for an objective."""
    response = llm.invoke(
        f"Create a day-by-day schedule for: {objective}"
    )
    return response.content

# ---- 4. CHECKLIST TOOL ----
def create_checklist(objective: str) -> str:
    """Create a clean and organized checklist."""
    response = llm.invoke(
        f"Create a detailed checklist for: {objective}"
    )
    return response.content

# ---- PACKAGE TOOLS INTO LANGCHAIN TOOL OBJECTS ----
tools = [
    Tool(name="Research",   func=research_topic,   description="Search for information online."),
    Tool(name="Planner",    func=create_plan,      description="Create detailed plans."),
    Tool(name="Scheduler",  func=generate_schedule, description="Generate multi-day schedules."),
    Tool(name="Checklist",  func=create_checklist, description="Create organized checklists."),
]


In [18]:
# ============================================================
# CREATE TRADITIONAL AGENT (OPTIONAL)
# ============================================================
# This is not BabyAGI yet – this is a reactive agent that can use tools.
# BabyAGI will call tools internally, but we show both for clarity.

agent = initialize_agent(
    tools=tools,
    llm=llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True,
    max_iterations=5
)


C:\Users\Adria\AppData\Local\Temp\ipykernel_16156\4101296421.py:7: LangChainDeprecationWarning: LangChain agents will continue to be supported, but it is recommended for new use cases to be built with LangGraph. LangGraph offers a more flexible and full-featured framework for building agents, including support for tool-calling, persistence of state, and human-in-the-loop workflows. For details, refer to the [LangGraph documentation](https://langchain-ai.github.io/langgraph/) as well as guides for [Migrating from AgentExecutor](https://python.langchain.com/docs/how_to/migrate_agent/) and LangGraph's [Pre-built ReAct agent](https://langchain-ai.github.io/langgraph/how-tos/create-react-agent/).
  agent = initialize_agent(


In [ ]:
# PREVIOUS CODE 
from langchain.agents import AgentExecutor, Tool, ZeroShotAgent
from langchain.chains import LLMChain
from langchain_community.utilities import SerpAPIWrapper
from langchain_openai import OpenAI

todo_prompt = PromptTemplate.from_template(
    "You are a planner who is an expert at coming up with a todo list for a given objective. Come up with a todo list for this objective: {objective}"
)
todo_chain = LLMChain(llm=OpenAI(temperature=0), prompt=todo_prompt)
search = SerpAPIWrapper()
tools = [
    Tool(
        name="Search",
        func=search.run,
        description="useful for when you need to answer questions about current events",
    ),
    Tool(
        name="TODO",
        func=todo_chain.run,
        description="useful for when you need to come up with todo lists. Input: an objective to create a todo list for. Output: a todo list for that objective. Please be very clear what the objective is!",
    ),
]


prefix = """You are an AI who performs one task based on the following objective: {objective}. Take into account these previously completed tasks: {context}."""
suffix = """Question: {task}
{agent_scratchpad}"""
prompt = ZeroShotAgent.create_prompt(
    tools,
    prefix=prefix,
    suffix=suffix,
    input_variables=["objective", "task", "context", "agent_scratchpad"],
)

/opt/anaconda3/lib/python3.11/site-packages/langchain_core/_api/deprecation.py:119: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 0.3.0. Use RunnableSequence, e.g., `prompt | llm` instead.
  warn_deprecated(


In [ ]:
# PREVIOUS CODE 
# llm = OpenAI(temperature=0)
llm_chain = LLMChain(llm=llm, prompt=prompt)
tool_names = [tool.name for tool in tools]
agent = ZeroShotAgent(llm_chain=llm_chain, allowed_tools=tool_names)
agent_executor = AgentExecutor.from_agent_and_tools(
    agent=agent, tools=tools, verbose=True
)

/opt/anaconda3/lib/python3.11/site-packages/langchain_core/_api/deprecation.py:119: LangChainDeprecationWarning: The class `ZeroShotAgent` was deprecated in LangChain 0.1.0 and will be removed in 0.2.0. Use create_react_agent instead.
  warn_deprecated(


### Run the BabyAGI

Now it's time to create the BabyAGI controller and watch it try to accomplish your objective.

In [19]:
# ============================================================
# CREATE BABYAGI INSTANCE
# ============================================================
# BabyAGI uses:
#  - The LLM
#  - A vectorstore to store tasks/results
#  - Automatically generated tasks
#  - max_iterations to prevent infinite loops

baby_agi = BabyAGI.from_llm(
    llm=llm,
    vectorstore=vectorstore,
    verbose=False,
    max_iterations=3
)


In [20]:
# ============================================================
# USER INTERFACE / INTERACTION
# ============================================================

print("\n" + "="*60)
print("🤖 AI PLANNING ASSISTANT (BabyAGI)")
print("="*60)

print("\nExamples you can try:")
print("  • Plan a 5-day trip to Italy")
print("  • Create a meal prep plan for next week")
print("  • Organize a product launch")
print("  • Plan a home renovation project")

user_objective = input("\nWhat would you like help planning?\n> ").strip()

# Default if user presses Enter
if not user_objective:
    user_objective = "Plan a 5-day trip to Italy"

print(f"\n{'*'*60}")
print(f"OBJECTIVE: {user_objective}")
print(f"{'*'*60}\n")



🤖 AI PLANNING ASSISTANT (BabyAGI)

Examples you can try:
  • Plan a 5-day trip to Italy
  • Create a meal prep plan for next week
  • Organize a product launch
  • Plan a home renovation project

************************************************************
OBJECTIVE: Plan a 5-day trip to Italy
************************************************************



In [ ]:
# PREVIOUS CODE 
# ============================================================
# RUN BABYAGI
# ============================================================
#try:
    #result = baby_agi.invoke({"objective": user_objective})

    #print(f"\n{'*'*60}")
    #print("✅ FINAL PLANNING OUTPUT")
    #print(f"{'*'*60}\n")
    #print(result)

#except Exception as e:
    #print("\n❌ ERROR OCCURRED:", e)
    #print("\nTROUBLESHOOTING:")
    #print("1. Are your API keys correct?")
    #print("2. Do you have internet access?")
    #print("3. Is SerpAPI installed? Run: pip install google-search-results")



*****TASK LIST*****

1: Make a todo list

*****NEXT TASK*****

1: Make a todo list

*****TASK RESULT*****

Day 1:
- Arrive in Italy
- Check into hotel
- Explore the local area
- Research nearby restaurants for dinner

Day 2:
- Visit historical sites (Colosseum, Roman Forum, Pantheon)
- Schedule guided tours if necessary
- Research local transportation options

Day 3:
- Take a day trip to Florence
- Visit the Uffizi Gallery and Ponte Vecchio
- Research best places to eat in Florence

Day 4:
- Explore the city of Venice
- Take a gondola ride
- Research must-see attractions in Venice

Day 5:
- Relax on the Amalfi Coast
- Visit Positano and Ravello
- Research beach clubs or activities for the day

Overall:
- Keep track of itinerary and reservations
- Stay organized with travel documents and maps
- Enjoy the local cuisine and culture of Italy.

*****TASK LIST*****

1: Explore the city of Milan
2: Visit the Duomo di Milano and Galleria Vittorio Emanuele II
3: Research best shopping spots in

In [ ]:
# PREVIOUS CODE 
# OBJECTIVE = "Write a weather report for SF today"

In [ ]:
# PREVIOUS CODE 

# Logging of LLMChains
#verbose = False
# If None, will keep on going forever
#max_iterations: Optional[int] = 3
#baby_agi = BabyAGI.from_llm(
    #llm=llm,
    #vectorstore=vectorstore,
    #task_execution_chain=agent_executor,
    #verbose=verbose,
    #max_iterations=max_iterations,
#)

In [ ]:
# PREVIOUS CODE 

#baby_agi({"objective": OBJECTIVE})

/opt/anaconda3/lib/python3.11/site-packages/langchain_core/_api/deprecation.py:119: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain 0.1.0 and will be removed in 0.2.0. Use invoke instead.
  warn_deprecated(



*****TASK LIST*****

1: Make a todo list

*****NEXT TASK*****

1: Make a todo list


> Entering new AgentExecutor chain...
Thought: I should create a todo list based on the given objective
Action: TODO
Action Input: Create a todo list for SF weather report
Observation: 

1. Research the current weather patterns in San Francisco.
2. Determine the sources for obtaining accurate weather information for the city.
3. Create a list of key weather elements to include in the report, such as temperature, precipitation, wind speed, and humidity.
4. Decide on the format of the report, whether it will be a written document, a presentation, or a combination of both.
5. Set a timeline for when the report needs to be completed and shared.
6. Gather data from reliable sources, such as the National Weather Service or local meteorologists.
7. Organize the data into a clear and easy-to-understand format.
8. Include any relevant graphics or visuals to enhance the report.
9. Proofread and edit the report 

{'objective': 'Write a weather report for SF today'}